<a href="https://colab.research.google.com/github/Oktora15/sentimen-analysis-shopee/blob/main/sentimen-analysis-shopee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-play-scraper pandas scikit-learn Sastrawi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re

from google_play_scraper import reviews, Sort
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [ ]:
result, _ = reviews(
    'com.shopee.id',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=3000
)

data = pd.DataFrame(result)
data = data[['content', 'score']]

In [ ]:
def label_sentimen(score):
    if score >= 4:
        return 'positif'
    elif score == 3:
        return 'netral'
    else:
        return 'negatif'

data['label'] = data['score'].apply(label_sentimen)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['clean_text'] = data['content'].apply(clean_text)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.85
)

X = tfidf.fit_transform(data['clean_text'])
y = data['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model_svm = SVC(kernel='linear', class_weight='balanced')
model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)
print("SVM:", accuracy_score(y_test, y_pred_svm))

SVM: 0.84


In [ ]:
model_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))

Logistic Regression: 0.8316666666666667


In [ ]:
model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

y_pred_nb = model_nb.predict(X_test)
print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))

Naive Bayes: 0.8616666666666667


In [ ]:
text = ["aplikasinya sangat bagus dan membantu"]

text_tfidf = tfidf.transform(text)

prediksi = model_svm.predict(text_tfidf)

print("Hasil prediksi:", prediksi)

Hasil prediksi: ['positif']


**Hasil Evaluasi Model**

Berdasarkan hasil pengujian pada dataset ulasan aplikasi Shopee, diperoleh nilai akurasi sebagai berikut:
1. Support Vector Machine (SVM): 0.853
2. Logistic Regression: 0.8383
3. Naive Bayes: 0.8583

Dari ketiga model yang diuji, Naive Bayes menunjukkan performa terbaik dengan akurasi tertinggi sebesar 0.8583

**Analisis Hasil**

Performa Naive Bayes lebih unggul dibandingkan model lainnya karena algoritma ini efektif dalam menangani data teks yang direpresentasikan dalam bentuk frekuensi kata.
Sementara itu, SVM dan Logistic Regression memberikan hasil yang cukup baik namun sedikit lebih rendah.
Perbedaan performa ini juga dapat dipengaruhi oleh distribusi data serta karakteristik dataset yang digunakan, termasuk ketidakseimbangan jumlah data pada masing-masing kelas.

**Kesimpulan**

Berdasarkan hasil eksperimen, model Naive Bayes dipilih sebagai model terbaik dalam melakukan klasifikasi sentimen karena memiliki akurasi paling tinggi dibandingkan model lainnya, yaitu sebesar 0.8583.
Model ini telah mampu melakukan klasifikasi sentimen dengan baik pada data ulasan pengguna.

Hasil ini menunjukkan bahwa metode machine learning dapat digunakan untuk menganalisis opini pengguna secara otomatis dari data teks.

**Penutup**

Model Naive Bayes dipilih sebagai model final untuk inference karena memiliki akurasi tertinggi dibandingkan model lainnya.